In [1]:
from __future__ import annotations
import torch
from typing import List, Tuple

from FlagEmbedding import FlagReranker, BGEM3FlagModel
from qdrant_client import QdrantClient

from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage

d:\Github_ThisPC\muict_thai_legal_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.rag.hybrid_retriever import HybridRetriever
from src.rag.hybridrag_langhchain import build_models

In [3]:
QDRANT_URL       = "http://localhost:6333"
EMBED_MODEL_NAME = "VISAI-AI/nitibench-ccl-human-finetuned-bge-m3"
RERANK_MODEL     = "BAAI/bge-reranker-v2-m3"

RETRIEVAL_LIMIT = 3  # candidates sent to reranker
RERANK_LIMIT = 3
# FINAL_LIMIT = 3  # top-k returned to LLM

In [4]:
embed_model, reranker, client = build_models()

retriever = HybridRetriever(
            embed_model=embed_model,
            reranker=reranker,
            client=client,
            retrieval_limit=RETRIEVAL_LIMIT,
            reranking_limit=RERANK_LIMIT,
        )

[RAG] Running on: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


In [ ]:
# from qdrant_client import QdrantClient
# from qdrant_client.http import models

# COLLECTION_NAME = "thai_laws_collection"


# def ref_law_link(list_pts): # List[ScorePoint from Qdrant]
#     if not list_pts:
#         return list_pts

#     first_context = list_pts[0]
#     query_lst = first_context.payload.get("reference_laws", [])

#     if not query_lst:
#         return list_pts

#     # Batch query: รวม filter ทุก ref_law ใน scroll เดียว (แก้ N+1 problem)
#     should_conditions = [
#         models.Filter(
#             must=[
#                 models.FieldCondition(
#                     key="law_name", match=models.MatchValue(value=ref["law_name"])
#                 ),
#                 models.FieldCondition(
#                     key="section_num", match=models.MatchValue(value=ref["section_num"])
#                 ),
#             ]
#         )
#         for ref in query_lst
#     ]

#     batch_results, _ = retriever.client.scroll(
#         collection_name=COLLECTION_NAME,
#         scroll_filter=models.Filter(
#             should=should_conditions
#         ),  # should รับ List[Filter] ได้
#         limit=len(query_lst),
#         with_payload=True,
#         with_vectors=False,
#     )

#     ref_scored_pts = [
#         models.ScoredPoint(
#             id=res.id,
#             payload=res.payload,
#             score=first_context.score, # use same score as first context
#             version=0,
#             vector=res.vector,
#         )
#         for res in batch_results
#     ]

#     return [first_context, *ref_scored_pts, *list_pts[1:]]

### Retrieve

In [5]:
# query = "มาตราใดจากประมวลกฎหมายแพ่งและพาณิชย์ไม่ถูกนำมาใช้บังคับกับบัตรเงินฝากแม้จะเป็นส่วนหนึ่งของข้อบังคับที่เกี่ยวข้อง"
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"

dense_vec, sparse_vec = retriever._encode_query(query)
candidates = retriever._hybrid_search(dense_vec, sparse_vec)
reranked_pts = retriever._rerank(query, candidates)
augmented_context = retriever._ref_law_link(reranked_pts)
# retrieve function
docs = [ # List[Langhchain Docs]
    Document(
        page_content=r.payload.get("text", ""),
        metadata={
            "law_name": r.payload.get("law_name", ""),
            "section_num": r.payload.get("section_num", ""),
            "reference_laws": r.payload.get("reference_laws", []),
            "rank": i,
            "score": r.score,
        },
    )
    for i,r in enumerate(augmented_context, start=1)
]


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
